# Infering Policy Rates from OIS Curves - Work in progress

A common practice among interest-rate practitioners is to use yield curve, discount factors and forward rate data — for example from bootstrapped OIS curves — to infer the market’s expectations of future policy rates, which can be interpreted as a short rate process. For instance, in the case of Chile’s Central Bank, this type of information is often illustrated in their monetary policy reports.

```{image} ./images/policyrate.png
:alt: Corredor TPM: Chile Banco Central
:class: bg-primary
:width: 500px
:align: center
```

### Model dependency

The discount factors and all other metrics are derived from a **risk-neutral** expectation. The random variable $\exp\{-\int_t^T r_s ds\}$ depends on the short rate process $ r_t $, which is not directly observable in the market. In fact, different dynamics for $r_t$ can lead to the same $P(t,T)$ (there are infinitely many short-rate models that can fit a given yield curve).

To illustrate this idea, let's build a numerical example:

- Data Generation:
  We start by creating a synthetic yield curve from the expected discount factors that arrise from multiple simulations of a short rate process. We want to build a dynamic that behaves as closely as posible to the evolution of the policy rate. For this, we define a pure-jump process as

  $$
      \int_t^T r_s\, ds 
      = r_t (T - t) + \sum_{\tau_i \in (t, T]} J_i (T - \tau_i),
  $$
  where $\bar{r}_n$ denotes the prevailing short rate during the period $[\tau_n,\tau_{n+1})$ and $J_n$ represents the stochastic jump in the policy rate at meeting $\tau_n$. The only source of randomness in this setting is the *policy decision itself*, not continuous micro-fluctuations of the short rate. 

- Model Setup:
  For the model, we will use a Hull-White one-factor model to represent the short-rate dynamics:
    $$
    dr_t = \alpha(\theta(t) - r_t) dt + \sigma dW_t^{\mathbb Q},
    $$
  where $\alpha$ is the speed of mean reversion, $\sigma$ is the volatility, and $\theta(t)$ is a time-dependent mean reversion level that we will calibrate to fit the observed yield curve.This functions is given by
    $$
    \theta(t) = \frac{\partial f(0,t)}{\partial t} + \alpha f(0,t) + \frac{\sigma^2}{2\alpha}(1 - e^{-2\alpha t}).
    $$  
  The expected value of $r_t$ under $\mathbb Q$ is
    $$
    \mathbb{E}^{\mathbb Q}[r_t] = r_0 e^{-\alpha t} + \int_0^t \alpha \theta(s) e^{-\alpha(t-s)} ds.
    $$
  Of course, different specifications of $r_t$ will produce different $\mathbb{E}^{\mathbb Q}[r_t]$. Notice that in this particular case the expectation is dependent on the parameters $a$ and $\sigma$, which clearly are not parameters required OIS swaps today.

In [93]:
from scipy.interpolate import UnivariateSpline
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import plotly
from IPython.display import display, HTML
from scipy.optimize import minimize, LinearConstraint
import pandas as pd 
from dataclasses import dataclass, field


### Simulation

We begin by simulating a policy rate process. We assume that there are quarterly meetings where the rate can jump up or down by a fixed amount, otherwise it remains constant. This is a simple way to mimic the behavior of central banks that adjust rates based on economic conditions.

In [ ]:
# Simulation parameters
YEARS = 10
STEPS_PER_YEAR = 360
DT = 1.0 / STEPS_PER_YEAR
R0 = np.array([0.03])
MEETING_DATES = np.arange(0.25, YEARS + 0.25, 0.25)  

# Jump size
J = 0.0025 
INTENSITIES = np.arange(-3*J,4*J, J)

# utility to generate probs that sum up to 1
def gen_probs(n_states: int):
    u = np.random.uniform(0, 1,size=n_states)
    return u/u.sum()

@dataclass
class Curve:
    times: np.array
    discounts: np.array

    def discount(self, t):
        return np.interp(t, self.times, self.discounts)
    
    def fwd_rate(self, t, T):
        df1 = self.discount(t)
        df2 = self.discount(T)
        return -np.log(df2/df1)/(T-t)
    
@dataclass
class Simulation:
    times: np.ndarray
    rate_paths: np.ndarray
    discount_paths: np.ndarray
    expected_rates: np.ndarray
    expected_discounts: np.ndarray
    probs: np.ndarray

    def get_curve(self):
        return Curve(self.times, self.expected_discounts)
    
@dataclass
class SimpleJumpSimulation:
    intensities: np.ndarray
    meeting_dates: np.ndarray
    dt: float
    r0: float
    same_prob: bool = True  

    def gen_paths(self, n_paths=10_000):
        horizon = float(self.meeting_dates[-1])
        steps = int(round(horizon / self.dt))
        times = np.arange(steps + 1) * self.dt  

        meeting_idx = np.round(self.meeting_dates / self.dt).astype(int)
        M = meeting_idx.size
        K = self.intensities.size

        increments = np.zeros((n_paths, steps + 1))
        probs = np.zeros((M, K))

        if self.same_prob:
            p_all = gen_probs(K)
            probs[:] = p_all
            for i, idx in enumerate(meeting_idx):
                increments[:, idx] = np.random.choice(self.intensities, size=n_paths, p=p_all)
        else:
            for i, idx in enumerate(meeting_idx):
                p_i = gen_probs(K)
                probs[i] = p_i
                increments[:, idx] = np.random.choice(self.intensities, size=n_paths, p=p_i)

        rate_paths = self.r0 + np.cumsum(increments, axis=1)

        acc_int_paths = np.zeros_like(rate_paths)
        acc_int_paths[:, 1:] = np.cumsum(rate_paths[:, :-1] * self.dt, axis=1)

        discount_paths = np.exp(-acc_int_paths)
        expected_rates = rate_paths.mean(axis=0)
        expected_discounts = discount_paths.mean(axis=0)

        return Simulation(times, rate_paths, discount_paths, expected_rates, expected_discounts, probs)

In [95]:
# Simulation
engine = SimpleJumpSimulation(INTENSITIES, MEETING_DATES, DT, R0)
n_paths = 1_000
simulation = engine.gen_paths()

show_k = 60  
idx_subset = np.random.choice(n_paths, size=min(show_k, n_paths), replace=False)

# Plotting
fig = make_subplots(rows=1, cols=2, subplot_titles=("Short-Rate Simulations", "Expected Discounts"))
for k in idx_subset:
    fig.add_trace(go.Scatter(x=simulation.times, y=simulation.rate_paths[k], mode='lines', line=dict(width=1, color='blue'), opacity=0.1, showlegend=False, hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Scatter(x=simulation.times, y=simulation.expected_rates, mode='lines', name='Expected Short Rate', line=dict(width=1, color='red')), row=1, col=1)
fig.add_trace(go.Scatter(x=simulation.times, y=simulation.expected_discounts, mode='lines',name='Expected Discounts', line=dict(width=1)),row=1, col=2)
fig.update_yaxes(tickformat=".2%", row=1, col=1, title_text="Rate")
fig.update_yaxes(row=1, col=2, title_text="Discount factor")
fig.update_xaxes(title_text="Time (years)", row=1, col=1)
fig.update_xaxes(title_text="Time (years)", row=1, col=2)
fig.update_layout(title="Short-Rate Simulations and Expected Discounts", showlegend=True, legend=dict(orientation="h", yanchor="top", y=-0.1, xanchor="center", x=0.5))
fig.show()

pd.DataFrame(simulation.probs, index=np.round(MEETING_DATES, 6), columns=[f"{s*1e4:+.0f} bp" for s in INTENSITIES])

,-75 bp,-50 bp,-25 bp,+0 bp,+25 bp,+50 bp,+75 bp,+100 bp
0.25,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
0.50,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
0.75,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
1.00,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
1.25,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
1.50,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
1.75,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
2.00,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
2.25,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376
2.50,0.097357,0.067179,0.012993,0.164307,0.153899,0.183064,0.164825,0.156376


In [96]:
eps = 1e-12
lnP_obs = np.log(simulation.expected_discounts)
f_raw = np.zeros_like(simulation.times)
f_raw[1:] = -(lnP_obs[1:] - lnP_obs[:-1]) / DT

s_factor = 5e-4
mask_pos = simulation.times > 0
spl_f = UnivariateSpline(simulation.times[mask_pos], f_raw[mask_pos], s=s_factor)
f = np.zeros_like(simulation.times)
f[mask_pos] = spl_f(simulation.times[mask_pos])
fprime = np.zeros_like(simulation.times)
fprime[mask_pos] = spl_f.derivative()(simulation.times[mask_pos])

r0 = f[1]

def hw_theta_from_curve(t, f, fprime, a, sigma):
    theta = np.zeros_like(t)
    theta[0] = f[1]  # benign at 0
    theta[1:] = fprime[1:] + a * f[1:] + (sigma**2)/(2*a) * (1.0 - np.exp(-2.0 * a * t[1:]))
    return theta

def hw_expectation_Q(t, theta, a, r0):
    E = np.zeros_like(t)
    E[0] = r0
    I = 0.0
    decay = np.exp(-a * DT)
    for k in range(1, len(t)):
        I = decay * I + a * 0.5 * DT * (theta[k] + decay * theta[k-1])
        E[k] = r0 * np.exp(-a * t[k]) + I
    return E

param_sets = [
    (0.20, 0.005),
    (0.50, 0.010),
    (1.00, 0.015),
]

cum_int_f = np.zeros_like(simulation.times)
cum_int_f[1:] = np.cumsum(0.5 * (f[:-1] + f[1:]) * DT)
P_fit = np.exp(-cum_int_f)
y_fit = np.zeros_like(simulation.times); y_fit[1:] = -np.log(P_fit[1:]) / simulation.times[1:]; y_fit[0] = r0

mean_paths = []
for a_hw, sigma_hw in param_sets:
    theta = hw_theta_from_curve(simulation.times, f, fprime, a_hw, sigma_hw)
    E_hw = hw_expectation_Q(simulation.times, theta, a_hw, r0)
    mean_paths.append((a_hw, sigma_hw, E_hw))


# to plot only some points
mask_t = (simulation.times * STEPS_PER_YEAR) % 180 == 0

# Plotting
fig = make_subplots(rows=1, cols=2, subplot_titles=("Discounts (observed vs fitted)", "E[r_t]"))
for a_hw, sigma_hw, _ in mean_paths:
    fig.add_trace(go.Scatter(x=simulation.times[mask_t], y=P_fit[mask_t], mode="markers+lines", name=f"Fitted-α={a_hw:.1f}, σ={sigma_hw:.2%}", line=dict(width=2, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=simulation.times[mask_t], y=simulation.expected_discounts[mask_t], name="Observed discounts", mode="lines", line=dict(color="green", width=1)), row=1, col=1)

for a_hw, sigma_hw, E_hw in mean_paths:
    fig.add_trace(go.Scatter(x=simulation.times, y=E_hw, mode="lines", name=f"E[r_t] — α={a_hw:.1f}, σ={sigma_hw:.2%}"), row=1, col=2)

for c in (1,2):
    fig.update_yaxes(tickformat=".2%", row=1, col=c)
    fig.update_xaxes(title_text="Time (years)", row=1, col=c)

fig.update_yaxes(title_text="Discount factor", row=1, col=1)
fig.update_yaxes(title_text="Short rate (mean)", row=1, col=2)
fig.update_layout(title="Hull-White Model", legend_title="Series")
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.3, xanchor="center", x=0.5))
fig.show()

As expected, the Hull-White model is able to fit perfectly the yield curve at time $t=0$, but $\mathbb{E}^{\mathbb Q}[r_t]$ varies significantly for different values of $\alpha$ and $\sigma$.

# Overcoming this limitations

As the dynamics of the short rate depend on our modeling choices, a possible approach that avoids this could be to learn from data the dynamics of the process, for example using a neural SDE model or other ML techniques. The problem of this is that, to my knowledge, there aren't yet methods that accodate jump dynamics in a sensible way. 

My take here is to model the evolution of the policy rate as close as possible to the actual behaviour of it, which surprisingly is not that complex in many geografies. Below is a sumary of the ideas behind a simplified model.

## A jump-only model for the short-rate process

We begin by considering the behavior of the short rate at the present time. 
Policy rates are known today and are adjusted by central banks only at scheduled monetary policy meetings. 
As a result, the short rate remains **constant** between meetings. 
When a new policy decision is announced, the rate changes in discrete steps, typically in multiples of a fixed increment $\delta$. 
We model this as

$$
    r_t = \bar{r}_n, \qquad 
    \bar{r}_n = \bar{r}_{n-1} + J_n, \qquad 
    \tau_n \le t < \tau_{n+1}, \qquad 
    \bar{r}_0 = r_0,
$$

where $\bar{r}_n$ denotes the prevailing short rate during the period $[\tau_n,\tau_{n+1})$ and $\{J_i\} = \{\dots, J^{\text{Up}}, J^{\text{Stay}}, J^{\text{Down}}, \dots\}$  represents the stochastic jump in the policy rate at meeting $\tau_n$. The only source of randomness in this setting is the *policy decision itself*, not continuous micro-fluctuations of the short rate. 
Consequently, no diffusion or volatility component is required. 
We also assume there are no unscheduled *surprise* meetings, so the sequence of meeting dates $\{\tau_n\}$ is known in advance. Under this specification, the integral of the short rate between $t$ and $T$ is

$$
    \int_t^T r_s\, ds 
    = r_t (T - t) + \sum_{\tau_i \in (t, T]} J_i (T - \tau_i),
$$

and lets define $\Delta \tau_i := T - \tau_i$ the length of time during which the new policy rate applies after the jump at $\tau_i$ and before maturity $T$. Then, the discount factor can be expressed as

$$
    P(t,T) 
        = e^{-r_t (T - t)} \cdot \mathbb{E}^{\mathbb{Q}}\left[e^{-\sum_{\tau_i \in (t, T]} J_i \Delta \tau_i} \Big| \mathcal{F}_t \right].
$$

Now, we have two options to proceed. If we assume that the jumps $\{J_i\}$ are independent and identically distributed (i.i.d.) random variables, we can get the expectation inside the product, making the calculation much easier. The problem with this approach is that we are assuming that the jumps do not depend on the last known rate, which is not realistic; we know that if a big hike just happened, the next jump is more likely to be smaller. If we assume that the jumps $\{J_i\}$ are not independent, we need to come up with a recursive way to compute the expectation. Let's see both cases.

### Independent Jumps

Assuming the jumps $\{J_i\}$ are conditionally independent given $\mathcal{F}_t$, the discount factor can be written as

$$
    P(t,T) 
    = \mathbb{E}^{\mathbb{Q}}\left[e^{-\int_t^T r_s\, ds} \Big| \mathcal{F}_t\right]
    = e^{-r_t (T - t)} \prod_{\tau_i \in (t, T]} \mathbb{E}^{\mathbb{Q}}\left[e^{-J_i \Delta \tau_¡} \Big| \mathcal{F}_t\right].
$$

Using the definition of instantaneous forward rate, for maturities $T$ that do not coincide with meeting dates, differentiation gives

$$
    f(t,T)
    = r_t-\sum_{\tau_i \in (t, T]} 
    \frac{\partial}{\partial T} \ln 
    \mathbb{E}^{\mathbb{Q}}\left[e^{-J_i \Delta \tau_¡} \Big| \mathcal{F}_t\right] = r_t + \sum_{\tau_i \in (t, T]} 
    \frac{\mathbb{E}^{\mathbb{Q}}\left[J_i\, e^{-J_i\Delta \tau_¡} \,\big|\, \mathcal{F}_t\right]}
    {\mathbb{E}^{\mathbb{Q}}\left[e^{-J_i\Delta \tau_¡ } \,\big|\, \mathcal{F}_t\right]}.
$$

### Dependent Jumps

Suppose the jumps $\{J_i\}$ are not independent (they depend on the previous policy rate desicion), then 
we can't factor the expectation as before, so we must compute it recursively using the tower property of conditional expectation:

$$
\mathbb{E}[X \mid \mathcal{F}_t] 
= \mathbb{E}\bigl[\mathbb{E}[X \mid \mathcal{F}_{\tau_i}]\big|\mathcal{F}_t\bigr],
\qquad t \le \tau_i.
$$

Applying this property to our discount factor, we have

$$
\begin{align*}
    P(t,T) 
    =& e^{-r_t (T - t)} \cdot \mathbb{E}^{\mathbb{Q}}\left[e^{-\sum_{\tau_i \in (t, T]} J_i \Delta \tau_i} \Big| \mathcal{F}_t \right] \\
    =& e^{-r_t (T - t)} \cdot \mathbb{E}^{\mathbb{Q}}\left[e^{-J_k \Delta \tau_k} \cdot \mathbb{E}^{\mathbb{Q}}\left[e^{-\sum_{\tau_i \in ( \tau_k, T]} J_i \Delta \tau_i} \Big| \mathcal{F}_{\tau_k}\right] \Big| \mathcal{F}_t \right]
\end{align*}
$$

where $k$ is the index of the first meeting date after $t$. This setup creates a recursive relation that we can solve via dynamic programming. Let's define the conditional expectation at meeting $\tau_k$ as
$$
u_k:= \mathbb{E}^{\mathbb{Q}}\left[e^{-J_i \Delta \tau_i} \cdot u_{k+1} \Big| \mathcal{F}_{\tau_k}\right], \qquad i \le k.
$$
Under this setup, we need to define some boundary conditions. For the last meeting that we want to match, say at $\tau_m$, we will impose 

$$
u_m = \mathbb{E}^{\mathbb{Q}}\left[e^{-J_m \Delta \tau_m}u_{m+1} \Big| \mathcal{F}_{\tau_m}\right] = \mathbb{1},
$$
wich implies no more jumps after $\tau_m$. At the other extreme-the first meeting after $t$-we have
$$
u_1 = \mathbb{E}^{\mathbb{Q}}\left[e^{-J_1 \Delta \tau_1} \cdot u_{2} \Big| \mathcal{F}_{\tau_1}\right].
$$

Of course this is a questionable assumption, but it allows us to close the recursion. In order to compute this expression, we can go backwards and compute:

$$
u_{1} = e^{-J_1 \Delta \tau_1} \cdot \mathbb{P}_0
$$

### We are not ready yet

After all this modeling, the idea would be to fit our model to the observed discount factors or forwards rates in order to infer the probabilities for each meeting. The problem is that without further assumptions, if we belive that there are more than two possible states for the jumps (e.g., up, down, stay), our model is underdetermined: there are infinite combinations of probabilities that can fit the same yield curve, so we need to impose further restrictions.

A simple way to overcome this issue is to use a maximunum entropy approach. What this approach tries to anwser is what's the best probability distributions that fits the mean of a random variable by imposing the least amount of information.

In [97]:
@dataclass
class IndependentJumpsModel:
    r0: np.ndarray
    meeting_dates: np.ndarray
    intensities: np.ndarray
    probs: np.ndarray = field(default=np.array)

    def __expectation(self, t):
        meetings_before_t = self.meeting_dates[self.meeting_dates < t]
        yf_meetings = t - meetings_before_t
        s = 0.0
        for i, yf in enumerate(yf_meetings):
            p = self.probs[i, :]
            w = np.exp(-yf * self.intensities)                   
            m0 = np.dot(p, w)                                    
            m1 = np.dot(p, self.intensities * w)                            
            s += m1 / m0
        return s

    def inst_fwd_rate(self, t):
        return self.r0 + self.__expectation(t)

    def discount(self, t):
        s = -self.r0 * t
        for i, tau_i in enumerate(self.meeting_dates):
            if tau_i < t - 1e-12:
                u = t - tau_i
                p = self.probs[i, :]
                m0 = np.dot(p, np.exp(-u * self.intensities))
                m0 = max(m0, 1e-18)
                s += np.log(m0)
        return np.exp(s)

    def fwd_rate(self, t, T):
        df1 = self.discount(t)
        df2 = self.discount(T)
        return -np.log(df2 / df1) / (T - t)

    def fit(self, curve: Curve, shift=0.1):
        knots = self.meeting_dates + shift
        M, K = self.meeting_dates.shape[0], self.intensities.shape[0]
        self.probs = np.zeros((M, K))

        cons = [{'type': 'eq', 'fun': lambda p: np.sum(p) - 1.0}]
        bnds = [(0.0, 1.0)] * K
        for i, knot in enumerate(knots):
            z0 = gen_probs(self.intensities.shape[0])
            def obj(p):                
                old = self.probs[i, :].copy()
                self.probs[i, :] = p
                err = curve.discount(knot) - self.discount(knot)
                self.probs[i, :] = old
                return err**2

            res = minimize(obj, z0, bounds=bnds, constraints=cons, options=dict(maxiter=10000, ftol=1e-15))
            self.probs[i, :] = res.x

    def get_probs(self):
        columns=[f"{s*1e4:+.0f} bp" for s in self.intensities]
        df = pd.DataFrame(self.probs, columns=columns, index=self.meeting_dates).round(2)
        return df
    
    def __repr__(self):
        return self.get_probs().to_string()
    
max_t = 2.2 # slightly higher t so we fit correctly up to t=2
fitted_meetings = MEETING_DATES[MEETING_DATES<max_t]
model = IndependentJumpsModel(R0, fitted_meetings, INTENSITIES)
curve = simulation.get_curve()

model.fit(curve, shift=0.15)
real_discounts = np.array([curve.discount(tt) for tt in fitted_meetings])
model_discounts = np.array([model.discount(tt) for tt in fitted_meetings])

# plotting
fig = go.Figure()
fig.add_trace(go.Scatter(x=fitted_meetings, y=real_discounts, mode='lines+markers', name='Real Discounts', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=fitted_meetings, y=model_discounts.flatten(), mode='lines+markers', name='Model Discounts', line=dict(color='red', dash='dash')))
fig.update_layout(title='Real vs Model Discounts', xaxis_title='Meeting Dates', yaxis_title='Discount Factor')
fig.show()

model.get_probs()

,-75 bp,-50 bp,-25 bp,+0 bp,+25 bp,+50 bp,+75 bp,+100 bp
0.25,0.11,0.14,0.14,0.00,0.02,0.08,0.26,0.25
0.50,0.10,0.13,0.00,0.08,0.09,0.20,0.27,0.12
0.75,0.06,0.14,0.08,0.08,0.09,0.17,0.22,0.17
1.00,0.08,0.00,0.09,0.23,0.11,0.20,0.18,0.12
1.25,0.10,0.05,0.01,0.23,0.17,0.15,0.05,0.24
1.50,0.08,0.00,0.15,0.08,0.23,0.13,0.20,0.13
1.75,0.10,0.07,0.05,0.07,0.24,0.12,0.21,0.15
2.00,0.00,0.13,0.08,0.14,0.16,0.23,0.06,0.19


In [98]:
def _as_list_of_arrays(arr_or_list, M, K):
    """Accepts an array of shape (M-1, K, K) or a list of M-1 (K,K) arrays and returns a list."""
    if isinstance(arr_or_list, list):
        assert len(arr_or_list) == M-1, "Need one transition matrix per transition (M-1)."
        for A in arr_or_list:
            assert A.shape == (K, K)
        return arr_or_list
    else:
        A = np.asarray(arr_or_list)
        assert A.shape == (M-1, K, K)
        return [A[i] for i in range(M-1)]

def _row_stochastic(A):
    A = np.maximum(A, 0.0)
    rowsum = A.sum(axis=1, keepdims=True)
    rowsum = np.where(rowsum <= 0, 1.0, rowsum)
    return A / rowsum

def build_homogeneous_transition(intensities: np.ndarray, kappa: float = 2.0, bias_stay: float = 0.0):
    K = intensities.size
    idx = np.arange(K)
    I, J = np.meshgrid(idx, idx, indexing="ij")
    S = -kappa * np.abs(I - J) + np.where(I == J, bias_stay, 0.0)
    S = S - S.max(axis=1, keepdims=True)
    P = np.exp(S)
    P = P / P.sum(axis=1, keepdims=True)
    return P


@dataclass
class DependentJumpsModel:
    r0: float
    meeting_dates: np.ndarray           # shape (M,)
    intensities: np.ndarray             # shape (K,)
    p1: np.ndarray                      # shape (K,), distribution of J_1
    transitions: List[np.ndarray]       # list of M-1 arrays (KxK): Π_1,...,Π_{M-1}
    eps: float = 1e-18

    def __post_init__(self):
        self.meeting_dates = np.asarray(self.meeting_dates, dtype=float)
        self.intensities   = np.asarray(self.intensities, dtype=float)
        self.p1            = np.asarray(self.p1, dtype=float)
        M = self.meeting_dates.shape[0]
        K = self.intensities.shape[0]
        assert self.p1.shape == (K,), "p1 must be a K-vector."
        self.p1 = np.maximum(self.p1, 0.0)
        s = self.p1.sum()
        self.p1 = self.p1 / (s if s > 0 else 1.0)
        self.transitions = _as_list_of_arrays(self.transitions, M, K)
        # ensure row-stochastic
        self.transitions = [_row_stochastic(A) for A in self.transitions]
        self.ones = np.ones(K)

    def _markov_weight_product(self, T: float) -> float:
        """
        Computes E[ exp(-∑ J_k Δ_k(T)) ] using the matrix product trick.
        Only meetings τ_k < T contribute. If none, returns 1.0.
        """
        idx = np.where(self.meeting_dates < T - 1e-12)[0]
        if idx.size == 0:
            return 1.0
        # meetings involved
        ks   = idx
        dlt  = (T - self.meeting_dates[ks])          # shape (m,)
        w    = np.exp(-np.outer(dlt, self.intensities))  # (m, K), row k has w_k
        # v_1 = p1 ⊙ w1
        v = self.p1 * w[0]
        # propagate: v ← v Π_1; then Hadamard with w_2; etc.
        for mpos, k in enumerate(ks[1:], start=1):
            v = v @ self.transitions[mpos-1]         # Π_{mpos}
            v = v * w[mpos]
        return float(v.sum())

    def discount(self, T: float) -> float:
        """P(0, T) under dependent jumps."""
        base = np.exp(-self.r0 * T)
        tail = self._markov_weight_product(T)
        return max(self.eps, base * tail)

    def fwd_rate(self, t: float, T: float) -> float:
        """Forward over [t, T]."""
        df1 = self.discount(t)
        df2 = self.discount(T)
        return -np.log(df2 / max(self.eps, df1)) / max(self.eps, (T - t))

    def inst_fwd_rate(self, T: float, h: float = 1e-3) -> float:
        """Instantaneous forward f(0,T) via symmetric difference of -ln P."""
        T1 = max(self.eps, T - h)
        T2 = T + h
        g1 = -np.log(self.discount(T1))
        g2 = -np.log(self.discount(T2))
        return (g2 - g1) / (T2 - T1)

    # --- Optional: calibrate only p1 given fixed transitions -----------------
    def fit_p1(self, curve, knots_shift: float = 0.15):
        """
        Fit the first meeting distribution p1 (only) to match early-knot discounts,
        keeping the transition matrices fixed. Useful when Π_k are specified from prior.
        """
        knots = self.meeting_dates + knots_shift
        K = self.intensities.size

        def objective(x):
            p = np.maximum(x, 0.0)
            s = p.sum()
            p = p / (s if s > 0 else 1.0)
            old = self.p1.copy()
            self.p1 = p
            err = 0.0
            for T in knots:
                err += (curve.discount(T) - self.discount(T))**2
            self.p1 = old
            return err

        # simplex constraints via softmax-like reparameterization using bounds
        x0 = np.ones(K) / K
        bnds = [(0.0, 1.0) for _ in range(K)]
        cons = [{'type': 'eq', 'fun': lambda x: np.sum(np.maximum(x,0.0)) - 1.0}]
        res = minimize(objective, x0, bounds=bnds, constraints=cons, options=dict(maxiter=2000, ftol=1e-16))
        self.p1 = np.maximum(res.x, 0.0)
        self.p1 /= self.p1.sum()

    def get_p1(self):
        cols = [f"{s*1e4:+.0f} bp" for s in self.intensities]
        return pd.DataFrame(self.p1.reshape(1,-1), columns=cols, index=["p1"]).round(4)

    def get_transition_df(self, k: int):
        """Return Π_k (1-based) as a DataFrame for inspection."""
        assert 1 <= k <= len(self.transitions)
        cols = [f"{s*1e4:+.0f} bp" for s in self.intensities]
        idx  = [f"prev {s*1e4:+.0f} bp" for s in self.intensities]
        return pd.DataFrame(self.transitions[k-1], columns=cols, index=idx).round(4)


# %% [markdown]
# ### Example usage on your synthetic curve
# We'll keep your simulation as-is, then (i) build a homogeneous mean-reverting
# transition Π, (ii) start from a diffuse p1, and (iii) optionally fit p1 to
# the early-meeting knots, keeping Π fixed (so the problem is well-posed).

# %%
# Reuse objects from your notebook section above:
#   - simulation (with .expected_discounts)
#   - Curve, MEETING_DATES, INTENSITIES, R0, etc.

curve = simulation.get_curve()

# meetings to fit/compare (same cut you used for the i.i.d. version)
max_t = 2.2
fitted_meetings = MEETING_DATES[MEETING_DATES < max_t]
M = fitted_meetings.size
K = INTENSITIES.size

# Build a homogeneous (time-stationary) mean-reverting Π and repeat it M-1 times
Pi0 = build_homogeneous_transition(INTENSITIES, kappa=2.0, bias_stay=0.5)
transitions = [Pi0 for _ in range(M-1)]

# Start from a neutral p1 (uniform). You could seed with a prior if you have one.
p1_0 = np.ones(K) / K

dep = DependentJumpsModel(
    r0=float(R0),
    meeting_dates=fitted_meetings,
    intensities=INTENSITIES,
    p1=p1_0,
    transitions=transitions
)

# Optionally fit only p1 to the early knots (Π fixed)
dep.fit_p1(curve, knots_shift=0.15)

# Compare discounts at the meeting knots
real_discounts = np.array([curve.discount(tt) for tt in fitted_meetings])
model_discounts = np.array([dep.discount(tt)   for tt in fitted_meetings])

fig = go.Figure()
fig.add_trace(go.Scatter(x=fitted_meetings, y=real_discounts, mode='lines+markers', name='Real Discounts', line=dict(width=2)))
fig.add_trace(go.Scatter(x=fitted_meetings, y=model_discounts, mode='lines+markers', name='Dependent-Jumps (Markov)', line=dict(width=2, dash='dash')))
fig.update_layout(title='Real vs Dependent-Jumps Model Discounts', xaxis_title='Meeting Dates', yaxis_title='Discount Factor', legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5))
fig.show()

# Inspect the fitted p1 and transition Π
dep.get_p1()
dep.get_transition_df(1)


/var/folders/cp/l432p0ns1t38qmnyr_3l3r000000gn/T/ipykernel_72901/3742532095.py:162: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



,-75 bp,-50 bp,-25 bp,+0 bp,+25 bp,+50 bp,+75 bp,+100 bp
prev -75 bp,0.9133,0.0750,0.0101,0.0014,0.0002,0.0000,0.0000,0.0000
prev -50 bp,0.0697,0.8496,0.0697,0.0094,0.0013,0.0002,0.0000,0.0000
prev -25 bp,0.0094,0.0691,0.8417,0.0691,0.0094,0.0013,0.0002,0.0000
prev +0 bp,0.0013,0.0093,0.0690,0.8406,0.0690,0.0093,0.0013,0.0002
prev +25 bp,0.0002,0.0013,0.0093,0.0690,0.8406,0.0690,0.0093,0.0013
prev +50 bp,0.0000,0.0002,0.0013,0.0094,0.0691,0.8417,0.0691,0.0094
prev +75 bp,0.0000,0.0000,0.0002,0.0013,0.0094,0.0697,0.8496,0.0697
prev +100 bp,0.0000,0.0000,0.0000,0.0002,0.0014,0.0101,0.0750,0.9133


![](#tbl:probs_fit)

## Appendix 1: A brief review of the math of interest rates

First, lets recall some **definitions** and results commonly used in interest rate modeling. The **discount factor** between time $t$ and $T$ is
$$
P(t,T) = \mathbb{E}^{\mathbb{Q}}\left[\exp\left(-\int_t^T r_s\, ds\right) \Big| \mathcal{F}_t\right],
$$
where $r_t$ is the **instantaneous short rate** and $\mathbb Q$ is the **risk-neutral measure**. The **instantaneous forward rate** with maturity $T$ observed at time $t$ is
$$
f(t,T) = -\frac{\partial}{\partial T} \ln P(t,T).
$$
The **market forward rate** between $T_1$ and $T_2$ is
$$
F(t;T_1,T_2) 
= \frac{1}{T_2 - T_1}\left(\frac{P(t,T_1)}{P(t,T_2)} - 1\right)
= \frac{1}{T_2 - T_1}\left(\exp\left(\int_{T_1}^{T_2} f(t,u)\, du\right) - 1\right).
$$

An OIS swap is an agreement where one party pays a fixed rate $K$ at times $T_1, T_2, \ldots, T_n$ and receives the floating overnight rate at the same times. The value of this contract at time $t$ is
$$
V_{\mathrm{OIS}}(t;K) = \sum_{i=1}^n \delta_i P(t,T_i)\big(F(t;T_{i-1},T_i) - K\big),
$$
where $\delta_i$ is the year fraction between $T_{i-1}$ and $T_i$. The fair fixed rate $K^*$ is the rate that makes the value of the swap zero at inception:
$$
V_{\mathrm{OIS}}(t;K^*) = 0.
$$

Using the par-swap relationship,
$$
K^*\sum_{i=1}^n \delta_i P(t,T_i) = 1 - P(t,T_n),
$$
we can solve for $P(t,T_n)$ recursively:
$$
P(t,T_n) = \frac{1 - K^* \sum_{i=1}^{n-1} \delta_i P(t,T_i)}{1 + K^* \delta_n}.
$$

Repeating this for all maturities yields the discount curve, from which forward rates can be obtained using the definitions above.
